# 03 — Modelización
## Fase 3 del TFM: Análisis del comportamiento electoral territorial en Colombia

Este notebook orquesta las funciones de `src/modelo.py`. El detalle completo del proceso de
decisión (qué se probó, qué se descartó y por qué) está en la bitácora interna de Fase 3
(`bitacora_fase3_modelizacion_v*.md`, no versionada en git). Aquí se documentan las decisiones
ya tomadas y se deja el pipeline formal ejecutado.

**Decisiones heredadas de Fase 1/2 que no se reabren:**
- Filtrar `valido_para_modelado == 1`.
- `peso_muestral` como `sample_weight` en todo ajuste y toda métrica.
- Reportar todo modelo con y sin `lag_pct_izquierda`.
- Validación por ventana expansiva temporal, nunca split aleatorio.
- Escalado siempre dentro de cada ventana (nunca sobre todo el dataset).

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import modelo as m

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

BASE_DIR = '..'  # configurable, sin rutas locales hardcodeadas

df = pd.read_csv(f'{BASE_DIR}/datos/procesados/dataset_maestro_electoral.csv')
print('Filas totales:', len(df))
df = df[df['valido_para_modelado'] == 1].copy()
print('Filas tras filtrar valido_para_modelado==1:', len(df))
df[['ano','pct_izquierda','lag_pct_izquierda','nbi_total','per_ocu']].groupby('ano').mean().round(2)

Filas totales: 5605
Filas tras filtrar valido_para_modelado==1: 5604


,pct_izquierda,lag_pct_izquierda,nbi_total,per_ocu
ano,,,,
2006,19.35,4.91,45.26,22123.10
2010,7.15,19.38,45.40,13194.62
2014,10.48,7.15,22.93,14443.85
2018,24.57,10.48,22.93,10219.49
2022,35.75,24.56,22.89,14703.78


## 1. Decisión de territorio: sin `region_dane` ni `departamento` en el modelo predictivo

Se evaluó empíricamente si añadir territorio como control categórico mejora el ajuste en
validación expansiva (Ridge, ventanas 2014/2018/2022, comparación exploratoria con alpha fijo
para aislar el efecto de las variables, antes de pasar a RidgeCV). La celda siguiente reproduce
esa comparación.

In [2]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

def construir_X_territorio(sub, dep_onehot=False, region_onehot=False):
    X = sub[m.COLUMNAS_PREDICTORAS_LAG].copy()
    if region_onehot:
        X = pd.concat([X, pd.get_dummies(sub['region_dane'], prefix='reg')], axis=1)
    if dep_onehot:
        X = pd.concat([X, pd.get_dummies(sub['departamento'], prefix='dep')], axis=1)
    return X

def r2_promedio_variante_territorio(dep_onehot=False, region_onehot=False):
    r2s = []
    for ano in m.VENTANAS_ESTABLES:
        train = df[df['ano'] < ano]
        test = df[df['ano'] == ano]
        Xtr = construir_X_territorio(train, dep_onehot, region_onehot)
        Xte = construir_X_territorio(test, dep_onehot, region_onehot).reindex(columns=Xtr.columns, fill_value=0)
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
        modelo_r = Ridge(alpha=1.0).fit(Xtr_s, train['pct_izquierda'], sample_weight=train['peso_muestral'])
        pred = modelo_r.predict(Xte_s)
        r2s.append(r2_score(test['pct_izquierda'], pred, sample_weight=test['peso_muestral']))
    return np.mean(r2s)

comparacion_territorio = pd.DataFrame({
    'variante': ['Base (lag+NBI+per_ocu)', '+ region_dane one-hot', '+ departamento 33 niveles'],
    'r2_medio_2014_2018_2022': [
        r2_promedio_variante_territorio(),
        r2_promedio_variante_territorio(region_onehot=True),
        r2_promedio_variante_territorio(dep_onehot=True),
    ]
})
comparacion_territorio

,variante,r2_medio_2014_2018_2022
0,Base (lag+NBI+per_ocu),-0.239976
1,+ region_dane one-hot,-0.420058
2,+ departamento 33 niveles,-0.752635


**Decisión**: sin territorio categórico en el modelo predictivo. `departamento` (33 niveles)
es consistentemente el peor — fragmenta el ajuste con el volumen de filas disponible por ventana
temprana. `region_dane` es inconsistente entre ventanas. Esto es coherente además con el diseño de
negocio: se quiere que la heterogeneidad territorial aflore como **residuo**, no que quede
absorbida como control del modelo.

## 2. La ventana 2006→2010 se reporta aparte (no se promedia)

Entrenar con una sola elección (2006, n=1.117) y predecir 2010 extrapola linealmente fuera de
rango: el `lag_pct_izquierda` de entrada en test (~19, viene de 2006) está muy por encima del
rango visto en train (~5, viene de 2002), y el modelo predice muy por encima del resultado real de
2010 (fuga de votos hacia Mockus, ya documentado en el EDA de Fase 2). No es un problema de las
variables — es que un shock de nivel nacional no se puede anticipar con una sola elección de
historia.

In [3]:
print('per_ocu y lag_pct_izquierda por año (train 2006 vs test 2010):')
print(df.groupby('ano')[['per_ocu','lag_pct_izquierda']].mean().round(2).loc[[2006,2010]])

per_ocu y lag_pct_izquierda por año (train 2006 vs test 2010):
       per_ocu  lag_pct_izquierda
ano                              
2006  22123.10               4.91
2010  13194.62              19.38


## 3. Baselines ingenuos — la referencia obligatoria

Si el modelo ajustado no le gana con claridad a "copiar el resultado anterior del municipio",
eso es un hallazgo en sí mismo (la inercia manda), no un fracaso del modelado.

In [4]:
baseline_lag = m.resumen_variante('Naive: copiar lag', lambda d,a,c,efecto_anio=None: m.baseline_copiar_lag(d,a), df, [])
baseline_media = m.resumen_variante('Naive: media train', lambda d,a,c,efecto_anio=None: m.baseline_media_train(d,a), df, [])

cols = ['ano_test','rmse','mae','r2_global','corr_intra_anio']
print('--- Naive: copiar lag ---')
print(baseline_lag[cols].round(3).to_string(index=False))
print('\n--- Naive: media de train ---')
print(baseline_media[cols].round(3).to_string(index=False))

--- Naive: copiar lag ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 17.017 12.744     -3.332            0.614
     2014  9.304  6.934     -0.612            0.376
     2018 22.498 15.100     -0.427            0.337
     2022 14.174 11.456      0.602            0.924

--- Naive: media de train ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 14.428 13.489     -2.114              NaN
     2014  7.773  6.325     -0.125              NaN
     2018 22.376 15.418     -0.412              NaN
     2022 30.208 21.961     -0.806              NaN


## 4. Modelos ajustados: Ridge/Lasso → RandomForest/SVR → XGBoost (opcional)

Todos con `sample_weight=peso_muestral`, escalado dentro de cada ventana (StandardScaler fit solo
en train), alpha por CV interna en train (RidgeCV/LassoCV). Efecto-año = dummies de `ano`
(Forma 1, `drop_first=True`), calculadas solo sobre train — nunca la Forma 2 (desviación respecto
a la media nacional del propio año de test), que tiene fuga real.

Se reportan las 4 métricas: RMSE, MAE, R² global (ponderados) y correlación intra-año (ponderada)
— esta última es la que decide si el modelo sirve, porque el R² global castiga sobre todo el
salto de nivel nacional entre elecciones, que no es lo que interesa para el diferenciador
(los residuos dentro de cada elección).

In [5]:
resultados = {}
resultados['1. Ridge (lag+NBI+per_ocu)'] = m.resumen_variante('Ridge', m.entrenar_evaluar_ridge, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=False)
resultados['2. Ridge + efecto-año'] = m.resumen_variante('Ridge+EA', m.entrenar_evaluar_ridge, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=True)
resultados['3. Lasso + efecto-año'] = m.resumen_variante('Lasso+EA', m.entrenar_evaluar_lasso, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=True)
resultados['1b. Ridge SIN lag (solo NBI+per_ocu)'] = m.resumen_variante('Ridge sin lag', m.entrenar_evaluar_ridge, df, m.COLUMNAS_PREDICTORAS_SIN_LAG, efecto_anio=False)
for nombre, tabla in resultados.items():
    print(f'--- {nombre} ---')
    print(tabla[cols].round(3).to_string(index=False))
    print()

--- 1. Ridge (lag+NBI+per_ocu) ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 26.165 24.014     -9.240            0.610
     2014  7.308  5.684      0.005            0.180
     2018 22.135 15.332     -0.382            0.146
     2022 28.565 20.909     -0.615            0.924

--- 2. Ridge + efecto-año ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 26.165 24.014     -9.240            0.610
     2014 10.054  8.893     -0.883            0.331
     2018 18.748 13.868      0.009            0.276
     2022 19.842 14.620      0.221            0.924

--- 3. Lasso + efecto-año ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 29.719 26.929    -12.211            0.614
     2014 10.734  9.607     -1.146            0.347
     2018 18.624 13.919      0.022            0.287
     2022 17.766 13.074      0.375            0.922

--- 1b. Ridge SIN lag (solo NBI+per_ocu) ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 14.428 13.489  

In [6]:
# Modelos principales: RandomForest y SVR (con y sin efecto-año), y XGBoost opcional
resultados['4. RandomForest (con lag)'] = m.resumen_variante('RF', m.entrenar_evaluar_random_forest, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=False)
resultados['4b. RandomForest + efecto-año'] = m.resumen_variante('RF+EA', m.entrenar_evaluar_random_forest, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=True)
resultados['5. SVR (con lag)'] = m.resumen_variante('SVR', m.entrenar_evaluar_svr, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=False)
resultados['5b. SVR + efecto-año'] = m.resumen_variante('SVR+EA', m.entrenar_evaluar_svr, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=True)
if m.XGBOOST_DISPONIBLE:
    resultados['6. XGBoost (con lag)'] = m.resumen_variante('XGB', m.entrenar_evaluar_xgboost, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=False)
    resultados['6b. XGBoost + efecto-año'] = m.resumen_variante('XGB+EA', m.entrenar_evaluar_xgboost, df, m.COLUMNAS_PREDICTORAS_LAG, efecto_anio=True)
for nombre, tabla in resultados.items():
    if nombre.startswith(('4','5','6')):
        print(f'--- {nombre} ---')
        print(tabla[cols].round(3).to_string(index=False))
        print()

--- 4. RandomForest (con lag) ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 25.557 23.404     -8.770            0.299
     2014 11.643  8.937     -1.525            0.143
     2018 22.243 16.086     -0.395            0.027
     2022 28.039 21.173     -0.556            0.301

--- 4b. RandomForest + efecto-año ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 25.557 23.404     -8.770            0.299
     2014 13.236 11.377     -2.263            0.455
     2018 18.703 15.256      0.013            0.195
     2022 18.464 13.620      0.325            0.748

--- 5. SVR (con lag) ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 18.578 17.135     -4.162            0.094
     2014  8.151  6.188     -0.238            0.027
     2018 24.356 17.328     -0.673           -0.183
     2022 31.403 23.364     -0.951            0.354

--- 5b. SVR + efecto-año ---
 ano_test   rmse    mae  r2_global  corr_intra_anio
     2010 18.578 17.135     -4.162     

## 5. Tabla comparativa final — promedio sobre ventanas estables (2014/2018/2022)

La ventana 2010 se excluye del promedio por el motivo explicado en la sección 2 (se reporta
por separado, no se descarta el dato, solo no se promedia).

In [7]:
filas_resumen = []
for nombre, tabla in resultados.items():
    prom = m.promedio_ventanas_estables(tabla)
    filas_resumen.append({'variante': nombre, **prom.to_dict()})

baseline_prom_lag = m.promedio_ventanas_estables(baseline_lag)
baseline_prom_media = m.promedio_ventanas_estables(baseline_media)
filas_resumen.insert(0, {'variante': '0b. Naive: media train', **baseline_prom_media.to_dict()})
filas_resumen.insert(0, {'variante': '0. Naive: copiar lag', **baseline_prom_lag.to_dict()})

tabla_final = pd.DataFrame(filas_resumen).sort_values('corr_intra_anio', ascending=False)
tabla_final = tabla_final[['variante','rmse','mae','r2_global','corr_intra_anio']]
tabla_final.round(3)

,variante,rmse,mae,r2_global,corr_intra_anio
0,0. Naive: copiar lag,15.325,11.163,-0.146,0.546
4,3. Lasso + efecto-año,15.708,12.200,-0.250,0.519
3,2. Ridge + efecto-año,16.215,12.460,-0.218,0.510
7,4b. RandomForest + efecto-año,16.801,13.418,-0.641,0.466
11,6b. XGBoost + efecto-año,17.424,14.036,-1.105,0.436
2,1. Ridge (lag+NBI+per_ocu),19.336,13.975,-0.330,0.417
9,5b. SVR + efecto-año,17.738,12.572,-0.295,0.327
10,6. XGBoost (con lag),19.471,14.780,-0.669,0.199
6,4. RandomForest (con lag),20.642,15.399,-0.825,0.157
5,1b. Ridge SIN lag (solo NBI+per_ocu),20.191,14.606,-0.448,0.077


## 6. Conclusión de la Fase 3 (modelo predictivo)

**Ningún modelo ajustado — ni lineal (Ridge, Lasso) ni no lineal (RandomForest, SVR, XGBoost),
con o sin efecto-año — le gana con claridad a copiar el resultado de la elección anterior del
propio municipio** en la métrica que importa (correlación intra-año, promedio de las ventanas
estables 2014/2018/2022). El naive "copiar lag" obtiene la correlación intra-año más alta de
todas las variantes probadas.

Esto es un hallazgo del proyecto, no un fracaso del modelado: **la inercia electoral es el
predictor dominante del voto municipal** (hallazgo 1 de los cuatro que sostienen la memoria).
Se probaron modelos no lineales (RandomForest, SVR, XGBoost) precisamente para poder afirmar en
la memoria, con rigor, que tampoco superan a la inercia — no como vía para rescatar el R², sino
para cerrar la puerta.

El modelo con lag + efecto-año (variante 2) es el que se usa como base para el análisis de
residuos del siguiente notebook (`04_residuos_y_anomalias.ipynb`): es de los que mejor
correlación intra-año obtiene entre los modelos ajustados (no el naive, porque el naive no genera
un residuo interpretable frente a una estructura — su "residuo" sería literalmente el cambio de
voto respecto a la elección anterior, que es otra cosa, ya explorada en el EDA).

**Próximo paso**: `04_residuos_y_anomalias.ipynb` — residuos del modelo con lag+efecto-año,
filtrados por `baja_confiabilidad_electoral`, como señal de municipios que rompen su propia
tendencia histórica (hallazgo honesto: cambio político genuino, no coacción — ver bitácora Fase 3
para el detalle completo de por qué se descartó la hipótesis de coacción).